<a href="https://colab.research.google.com/github/yeogyung/welfare-resource-finder/blob/main/20%EA%B8%B0_%EB%8F%84%EC%A0%84%ED%95%99%EA%B8%B0%EC%A0%9C_%EB%B0%95%EC%97%AC%EA%B2%BD_RAG_%ED%94%84%EB%A1%9C%ED%86%A0%ED%83%80%EC%9E%85_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. 패키지 설치

In [ ]:
!pip install transformers accelerate bitsandbytes sentence-transformers faiss-cpu pandas openpyxl -q

2. 라이브러리 임포트

In [ ]:
import pandas as pd
import numpy as np
import torch
import faiss
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

print("라이브러리 로드 완료")

In [ ]:
from transformers import BitsAndBytesConfig, AutoConfig

3. 노인복지 자원DB_엑셀 파일 업로드

In [ ]:
from google.colab import files

uploaded = files.upload()

3-1. 시트명 확인

In [ ]:
excel_file = pd.ExcelFile("노인복지자원DB_MVP_v1.xlsx")
print(excel_file.sheet_names)

4. DB 로드 + 검색 텍스트 생성

In [ ]:
# 1) 엑셀 로드
df = pd.read_excel(
    "노인복지자원DB_MVP_v1.xlsx",
    sheet_name="노인복지DB_v1"
)

print(f"DB 로드 완료: {len(df)}건")
print(df[['ID', '구분', '카테고리', '서비스명']].head(10))

# 2) 검색용 텍스트 / 노인 친화 설명 텍스트 생성
def make_search_text(row):
    return (
        f"서비스명: {row['서비스명']} | "
        f"카테고리: {row['카테고리']} | "
        f"지원대상: {row['지원 대상']} | "
        f"설명: {row['서비스 요약 (노인 친화)']}"
    )

def make_friendly_text(row):
    contact = (
        f" 문의: {row['문의처']}"
        if pd.notna(row['문의처']) and str(row['문의처']).strip()
        else ""
    )
    return (
        f"[{row['서비스명']}]\n"
        f"{row['서비스 요약 (노인 친화)']}\n"
        f"대상: {row['지원 대상']}{contact}"
    )

df['search_text'] = df.apply(make_search_text, axis=1)
df['friendly_text'] = df.apply(make_friendly_text, axis=1)
print("검색 텍스트 생성 완료")

6. 임베딩 + FAISS

In [ ]:
print("임베딩 모델 로드 중... (1~2분)")
embed_model = SentenceTransformer("jhgan/ko-sroberta-multitask")

print("임베딩 생성 중... (1~2분)")
embeddings = embed_model.encode(
    df['search_text'].tolist(),
    show_progress_bar=True,
    batch_size=32
)
embeddings = np.array(embeddings).astype('float32')

faiss.normalize_L2(embeddings)
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print(f"FAISS 인덱스 구축 완료 ({index.ntotal}건)")

7. 검색 함수

In [ ]:
def search_welfare(query, top_k=3):
    query_vec = embed_model.encode([query]).astype('float32')
    faiss.normalize_L2(query_vec)
    scores, indices = index.search(query_vec, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        row = df.iloc[idx]
        results.append({
            'name': row['서비스명'],
            'category': row['카테고리'],
            'friendly_text': row['friendly_text'],
            'score': float(score)
        })
    return results

print("=== 검색 테스트 ===")
for r in search_welfare("혼자 사는데 쓰러지면"):
    print(f"[유사도 {r['score']:.3f}] {r['name']} ({r['category']})")

8. LLM(EEVE) 로드

In [ ]:
MODEL_ID = "yanolja/EEVE-Korean-Instruct-2.8B-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)
config.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    config=config,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=600,
    temperature=0.7,
    do_sample=True,
    repetition_penalty=1.1
)

print("EEVE 모델 로드 완료")

9. RAG 추천 함수

In [ ]:
# 노인 친화적 답변을 위한 시스템 가이드라인 설정
SYSTEM_BASE = """당신은 한국 노인이 복지 자원을 쉽게 찾을 수 있도록 도와주는 친절한 안내 도우미입니다.
[규칙]
1. 아래 제공된 복지 자원 목록에서만 답변합니다.
2. 전문 용어 대신 쉬운 말을 씁니다. (예: "재가급여" -> "집으로 와서 도와드리는 서비스")
3. 각 자원을 2~3문장으로 설명하고, 전화번호나 신청 방법을 꼭 포함합니다.
4. 목록에 없는 정보는 "주민센터나 129에 확인해 보세요"로 안내합니다.
5. 따뜻하고 존댓말로 대화합니다.
"""

def rag_recommend(user_input, top_k=3):

    retrieved = search_welfare(user_input, top_k=top_k)

    resource_text = "\n\n".join([r['friendly_text'] for r in retrieved])
    system_prompt = f"{SYSTEM_BASE}\n\n[검색된 관련 복지 자원]\n{resource_text}"

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    output = pipe(prompt)[0]["generated_text"]
    response = output[len(prompt):].strip()

    return {
        "question": user_input,
        "retrieved": [r['name'] for r in retrieved],
        "answer": response
    }

print("RAG 추천 시스템 준비 완료")

* RAG Test 1

In [ ]:
result = rag_recommend("혼자 사는데 갑자기 쓰러지면 어떡하죠?")
print(f"📌 검색된 자원: {', '.join(result['retrieved'])}")
print(f"\n💬 답변:\n{result['answer']}")

* RAG Test 2

In [ ]:
result = rag_recommend("매달 생활비가 부족해요. 집은 있어요.")
print(f"📌 검색된 자원: {', '.join(result['retrieved'])}")
print(f"\n💬 답변:\n{result['answer']}")

* RAG Test 3

In [ ]:
result = rag_recommend("뭔가 배우거나 활동하고 싶은데 어디 가면 돼요?")
print(f"📌 검색된 자원: {', '.join(result['retrieved'])}")
print(f"\n💬 답변:\n{result['answer']}")

9.2. RAG 성능 개선안_v2 (Strict Prompting & Few-shot 적용)

In [ ]:
# Few-shot 및 출력 제약 조건 강화
IMPROVED_SYSTEM_BASE = """당신은 어르신을 위한 복지 안내원입니다.
반드시 아래 제공된 [검색된 복지 자원]의 정보만 사용하여 다음 양식으로 답변하세요.

[답변 양식]
1. 인사말: (상황에 공감하는 짧은 인사)
2. 추천 서비스: (서비스명)
3. 상세 내용: (어르신이 이해하기 쉬운 2~3문장 설명)
4. 신청 방법: (지원 대상 및 문의처 정보)

[주의사항]
- 제공된 정보에 없는 전화번호나 신청 서류(여권 등)는 절대 지어내지 마세요.
- 모르는 내용은 "주민센터(129)에 문의하세요"라고 안내하세요.

"""

def rag_recommend_v2(user_input, top_k=2):
    retrieved = search_welfare(user_input, top_k=top_k)

    resource_text = "\n\n".join([r['friendly_text'] for r in retrieved])

    system_prompt = f"{IMPROVED_SYSTEM_BASE}\n\n[검색된 복지 자원]\n{resource_text}"

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    output = pipe(prompt)[0]["generated_text"]
    response = output[len(prompt):].strip()

    return response

print("=== [개선안] RAG v2 테스트 실행 ===")
test_query = "혼자 사는데 갑자기 쓰러지면 어떡하죠?"
print(f"질문: {test_query}")
print(f"답변:\n{rag_recommend_v2(test_query)}")

>> 답변에 환각이 심각한 수준

9.3. RAG 성능 개선안_v3 (Strict Constraints & url 포함)

In [ ]:
# 가독성 개선, 근거 강화_url 포함
def rag_recommend_v3(user_input, top_k=2):

    retrieved = search_welfare(user_input, top_k=top_k)

    final_output = []

    for i, r in enumerate(retrieved):
        row_data = df[df['서비스명'] == r['name']].iloc[0]

        url_val = row_data.iloc[12] if len(row_data) > 12 else "주민센터 문의"

        if pd.isna(url_val) or str(url_val).strip() == "":
            url_val = "공식 홈페이지 확인 필요 (129 문의)"

        detail_info = r['friendly_text'].split('대상:')[0].replace('['+r['name']+']', '').strip()

        item_template = f"""
[{i+1}] {r['name']} ({r['category']})
▶ 서비스 설명: {detail_info}
▶ 누가 받을 수 있나요?: {r['friendly_text'].split('대상:')[-1].strip()}
▶ 상세 확인 및 신청(URL): {url_val}
        """
        final_output.append(item_template)

    STRICT_PROMPT = f"""당신은 노인복지 전문가입니다.
어르신이 "{user_input}"라고 물어보셨습니다.
어르신의 마음을 위로하고 안심시켜드리는 따뜻한 첫 인사말을 한 문장으로만 다정하게 작성하세요.
성별을 특정하지 말고, 반말을 쓰지 마세요. '어르신'이라는 명칭으로 답을 하세요.
'신뢰도'나 '답변 양식' 같은 단어는 절대 쓰지 마세요.

인사말:"""

    output = pipe(STRICT_PROMPT, max_new_tokens=100, temperature=0.1, do_sample=False)[0]["generated_text"]

    llm_intro = output.split("인사말:")[-1].strip()

    llm_intro = llm_intro.split('\n')[0].replace("(인사말 입력)", "").strip()

    full_response = f"💬 {llm_intro}\n" + "\n".join(final_output)

    return full_response

# 테스트 실행
print("=== [개선안] RAG v3 테스트 실행 ===")
print(rag_recommend_v3("혼자 사는데 갑자기 쓰러지면 어떡하죠?"))

* 시나리오 3가지 테스트

In [ ]:
scenarios = [
    "매달 생활비가 부족해요. 집은 있어요.",
    "컴퓨터를 배우고 싶은데 어디로 가야 하나요?",
    "몸이 안 좋은데 혼자 있다가 쓰러질까 봐 걱정돼요."
]

for q in scenarios:
    print(f"\n🚀 시나리오 테스트: {q}")
    print(rag_recommend_v3(q))
    print("="*60)

> 시나리오 3은 임베딩 모델의 유사도 판단에 따라 의료 지원이 우선 추천되었으나, 향후 키워드 가중치 설정을 통해 응급 서비스를 우선 배치할 예정.